# The virtual NEOARM: its ground truth, worked out once

The virtual NEOARM is TemGym's simulated microscope with hidden projection lenses inside and the
real NEOARM's lens table on its console. This notebook works it out once and writes everything to
**`virtual_neoarm.json`**, the one ground truth the rehearsals read:

1. the recorded NEOARM lens table (MAG, 40k–600k), used as it is;
2. hidden lenses fitted to it: with the objective as in the table, every recorded row images at
   its nominal magnification and within its depth of field;
3. DIFF rows designed on those lenses (the recorded table has none).

All DAC values are at 200 kV; a rehearsal at another HT scales them by the beam's momentum. Rerun
this notebook only to change the virtual NEOARM.

In [1]:
import json
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
from scipy.optimize import least_squares

from temgym_core.components import HardEdgeSolenoid, Plane
from temgym_core.pyjem_simulator import LensDrive
from temgym_core.run import position_and_angle_response

jax.config.update("jax_enable_x64", True)
LENSES = ("IL1", "IL2", "IL3", "PL1")

## 1. The recorded NEOARM lens table

IL1, IL2, IL3, PL1 (and the objective, OLc and OLf) at each MAG row, read off the real NEOARM.
PL1 and the objective are the same in every row. The magnifications are the console's nominal
labels. A lens's relative strength is its DAC value in units that put the 40k row at
`STRENGTHS_40K` (only a choice of unit: the hidden lenses' gains absorb it).

In [2]:
raw = json.loads(Path("neoarm_lens_table.json").read_text())["raw_dac_hex"]
MAGNIFICATIONS = sorted(m for m in map(int, raw) if 40000 <= m <= 600000)
mag_rows = {m: {n: int(raw[str(m)][n], 16) for n in LENSES} for m in MAGNIFICATIONS}
objective = {n: int(raw["40000"][n], 16) for n in ("OLc", "OLf")}
STRENGTHS_40K = np.array([1.0, 0.6156826056827933, 0.5675232215801136, 1.7562259961484548, 0.8387606122572018])
drives = {n: LensDrive(mag_rows[40000][n] / u) for n, u in zip(LENSES, STRENGTHS_40K[1:])}
strengths = lambda row: np.r_[1.0, [drives[n].control(row[n]) for n in LENSES]]
U = np.array([strengths(mag_rows[m]) for m in MAGNIFICATIONS])
for m in MAGNIFICATIONS:
    print(f"{m:7d}x: {mag_rows[m]}")
print("objective:", objective)

  40000x: {'IL1': 41782, 'IL2': 24862, 'IL3': 35544, 'PL1': 64000}
  50000x: {'IL1': 42612, 'IL2': 25752, 'IL3': 34231, 'PL1': 64000}
  60000x: {'IL1': 43024, 'IL2': 27208, 'IL3': 32601, 'PL1': 64000}
  80000x: {'IL1': 43242, 'IL2': 29058, 'IL3': 30674, 'PL1': 64000}
 100000x: {'IL1': 43008, 'IL2': 32070, 'IL3': 29189, 'PL1': 64000}
 120000x: {'IL1': 42964, 'IL2': 32070, 'IL3': 27775, 'PL1': 64000}
 150000x: {'IL1': 42849, 'IL2': 33841, 'IL3': 26086, 'PL1': 64000}
 200000x: {'IL1': 42693, 'IL2': 38267, 'IL3': 23626, 'PL1': 64000}
 250000x: {'IL1': 42664, 'IL2': 38267, 'IL3': 21803, 'PL1': 64000}
 300000x: {'IL1': 42504, 'IL2': 40646, 'IL3': 19368, 'PL1': 64000}
 400000x: {'IL1': 42372, 'IL2': 44300, 'IL3': 15894, 'PL1': 64000}
 500000x: {'IL1': 42269, 'IL2': 47428, 'IL3': 12904, 'PL1': 64000}
 600000x: {'IL1': 42192, 'IL2': 50119, 'IL3': 10097, 'PL1': 64000}
objective: {'OLc': 45818, 'OLf': 39178}


## 2. The hidden lenses, fitted to the recorded table

The objective, IL1, IL2, IL3 and PL1 as five TemGym solenoids, then the camera: the model family
the stage-1 fit uses, 16 numbers (6 log gaps, 5 log lengths, the objective's strength, 4 log
gains). With the objective as in the table, they are fitted so that at every recorded row

* |A|, the magnification, matches the row's label (to about 1 %);
* the image is in focus to well within the depth of field. B / A is the defocus at the specimen;
  with a near-parallel beam it blurs the image by defocus × the beam's spread of angles, so the
  depth of field is (one camera pixel at the specimen) / (spread of angles): about 6 µm at 40k,
  0.4 µm at 600k, for 110 µm pixels and a 0.47 mrad rms spread;

with sizes kept physical: the specimen-to-camera length as before (0.72 m), gaps 20–400 mm (the
first 0.5–5 mm, the last 100–600 mm), lens lengths 0.2–30 mm. It is started from the earlier
made-up lenses and from 15 random changes of them (fixed seed), and the best kept. The labels are
nominal: the table itself has the same IL2 at 200k and 250k, so both cannot be exact.

In [3]:
SIGNS = jnp.array([1.0, 1.0, 1.0, -1.0])                                     # PL1 turns the beam the other way
P_MADE_UP = np.array([-6.16581793425276, -1.5677013503382955, -2.63169766451187, -2.855970331178832,
                      -2.9564473297712337, -1.1703236621139543, -7.824046010856292, -5.298317366548036,
                      -5.298317366548036, -5.298317366548036, -6.907755278982137, -0.4257450165772056,
                      -0.5978370007556204, -0.4307829160924542, -0.7985076962177716, -0.2876820724517809])
TOTAL = float(np.exp(P_MADE_UP[:11]).sum())
M = np.array(MAGNIFICATIONS, float)
DEPTH_OF_FIELD = 110e-6 / M / (1.1e-3 / 2.3548)                              # m, at each row


def lens_column(p, strengths):
    # five solenoids and the camera; p: log gaps (6), log lengths (5), objective q, log gains (4)
    gaps, lengths = jnp.exp(p[:6]), jnp.exp(p[6:11])
    q = jnp.r_[p[11] * strengths[0], SIGNS * jnp.exp(p[12:16]) * jnp.asarray(strengths[1:])]
    entrances = jnp.cumsum(gaps[:5]) + jnp.r_[0.0, jnp.cumsum(lengths[:4])]
    return [*(HardEdgeSolenoid(z=entrances[i], length=lengths[i], q=q[i]) for i in range(5)),
            Plane(z=gaps.sum() + lengths.sum())]


ab_rows = jax.vmap(lambda p, u: position_and_angle_response(lens_column(p, u)), in_axes=(None, 0))


def residuals(p):
    a, b = ab_rows(p, jnp.asarray(U))
    length = jnp.exp(p[:11]).sum()
    return jnp.concatenate([(b / a).real / DEPTH_OF_FIELD, 100 * jnp.log(jnp.abs(a) / M),
                            0.03 * (p - P_MADE_UP), jnp.array([100 * jnp.log(length / TOTAL)])])


lo = np.r_[np.log([0.0005, 0.02, 0.02, 0.02, 0.02, 0.1]), np.log([0.0002, 0.001, 0.001, 0.001, 0.0005]), -3, -4, -4, -4, -4]
hi = np.r_[np.log([0.005, 0.4, 0.4, 0.4, 0.4, 0.6]), np.log([0.002, 0.03, 0.03, 0.03, 0.01]), 3, 4, 4, 4, 4]
f, jac = jax.jit(residuals), jax.jit(jax.jacfwd(residuals))
rng = np.random.default_rng(0)
starts = [P_MADE_UP] + [P_MADE_UP + rng.normal(0, [0.3] * 11 + [0.2] + [0.5] * 4) for _ in range(15)]
fits = [least_squares(lambda p: np.asarray(f(p)), np.clip(s, lo + 1e-6, hi - 1e-6), jac=lambda p: np.asarray(jac(p)),
                      bounds=(lo, hi), max_nfev=3000) for s in starts]
P = min(fits, key=lambda fit: fit.cost).x

a, b = (np.asarray(v) for v in ab_rows(P, jnp.asarray(U)))
for m, ai, bi, dof in zip(MAGNIFICATIONS, a, b, DEPTH_OF_FIELD):
    defocus = (bi / ai).real
    print(f"{m:7d}x: true magnification {abs(ai):8.0f} ({abs(ai) / m - 1:+.1%}), defocus {defocus * 1e6:+.3f} µm "
          f"({abs(defocus) / dof:.2f} × depth of field)")
gaps, lengths = np.exp(P[:6]), np.exp(P[6:11])
centres = np.cumsum(gaps[:-1]) + np.r_[0, np.cumsum(lengths[:-1])] + lengths / 2
print("lens centres from the specimen (mm):", dict(zip(("objective", *LENSES), np.round(centres * 1e3, 1).tolist())),
      f"camera at {(gaps.sum() + lengths.sum()) * 1e3:.0f} mm")

  40000x: true magnification    40096 (+0.2%), defocus -0.265 µm (0.04 × depth of field)
  50000x: true magnification    49390 (-1.2%), defocus +0.195 µm (0.04 × depth of field)
  60000x: true magnification    61259 (+2.1%), defocus +0.393 µm (0.10 × depth of field)
  80000x: true magnification    80266 (+0.3%), defocus +0.488 µm (0.17 × depth of field)
 100000x: true magnification    99277 (-0.7%), defocus +0.363 µm (0.15 × depth of field)
 120000x: true magnification   117654 (-2.0%), defocus +0.335 µm (0.17 × depth of field)
 150000x: true magnification   146647 (-2.2%), defocus +0.272 µm (0.17 × depth of field)
 200000x: true magnification   211874 (+5.9%), defocus +0.188 µm (0.16 × depth of field)
 250000x: true magnification   241775 (-3.3%), defocus +0.171 µm (0.18 × depth of field)
 300000x: true magnification   303683 (+1.2%), defocus +0.083 µm (0.11 × depth of field)
 400000x: true magnification   406975 (+1.7%), defocus +0.009 µm (0.02 × depth of field)
 500000x: true magnif

## 3. The DIFF rows

In DIFF, IL1–IL3 image the objective's back focal plane (instead of its image plane) onto PL1's
object plane, so the diffraction pattern lands on the camera. For the whole column: a shift at the
specimen does not move the beam (A = 0) and a tilt of θ moves it L × θ (|B| = L, the camera
length). PL1 stays at its MAG value. Three lenses and two conditions leave one free direction: the
rows keep IL2 and IL3 close to the 40k row, so IL1 does the switching, as on the real NEOARM. They
are solved from the shortest camera length to the longest, each starting from the last, within
2000–63000, then rounded to the whole DAC values nearby with the smallest |A|.

In [4]:
CAMERA_LENGTHS = (0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0)       # m, JEOL's nominal steps
ab = jax.jit(lambda u: position_and_angle_response(lens_column(P, u)))
a_b = lambda row: tuple(complex(v) for v in ab(strengths(row)))
base = strengths(mag_rows[40000])[1:]
per_unit = np.array([drives[n].dac_per_unit for n in LENSES[:3]])
bounds = np.log(np.array([2000.0, 63000.0])[:, None] / (per_unit * base[:3]))
steps = np.stack(np.meshgrid(*[np.arange(-2, 3)] * 3, indexing="ij"), -1).reshape(-1, 3)


def diff_residuals(x, length):
    a, b = (complex(v) for v in ab(np.r_[1.0, base[:3] * np.exp(x), base[3]]))
    return np.r_[(a * np.conj(b)).real / abs(b) / 100, 50 * np.log(abs(b) / length), x[1:], 0.1 * x[:1]]


x, diff_rows = np.log([0.6, 1.0, 1.0]), {}
a, b = a_b(mag_rows[40000])
print(f"40k row: |A| = {abs(a):.0f}, |B| = {abs(b):.1e} m/rad")
for length in CAMERA_LENGTHS:
    x = least_squares(diff_residuals, np.clip(x, *(bounds + [[1e-3], [-1e-3]])), args=(length,), x_scale=0.1,
                      bounds=bounds).x
    dac = np.array([drives[n].dac(u) for n, u in zip(LENSES, np.r_[base[:3] * np.exp(x), base[3]])])
    nearby = [dict(zip(LENSES, (int(v) for v in dac + np.r_[s, 0]))) for s in steps]
    diff_rows[length] = min(nearby, key=lambda row: abs(a_b(row)[0]))
    a, b = a_b(diff_rows[length])
    print(f"{length * 100:5.0f} cm: {diff_rows[length]}   |A| = {abs(a):.2f}, |B| = {abs(b):.4f} m/rad")

40k row: |A| = 40096, |B| = 1.1e-02 m/rad


   15 cm: {'IL1': 29877, 'IL2': 25583, 'IL3': 37384, 'PL1': 64000}   |A| = 0.06, |B| = 0.1501 m/rad
   20 cm: {'IL1': 29997, 'IL2': 25229, 'IL3': 36561, 'PL1': 64000}   |A| = 0.02, |B| = 0.1999 m/rad
   25 cm: {'IL1': 30073, 'IL2': 24887, 'IL3': 35622, 'PL1': 64000}   |A| = 0.86, |B| = 0.2501 m/rad
   30 cm: {'IL1': 30126, 'IL2': 24596, 'IL3': 34552, 'PL1': 64000}   |A| = 0.01, |B| = 0.3001 m/rad
   40 cm: {'IL1': 30200, 'IL2': 24517, 'IL3': 31945, 'PL1': 64000}   |A| = 1.90, |B| = 0.3999 m/rad
   50 cm: {'IL1': 30243, 'IL2': 25630, 'IL3': 29376, 'PL1': 64000}   |A| = 2.56, |B| = 0.5000 m/rad
   60 cm: {'IL1': 30260, 'IL2': 27292, 'IL3': 27582, 'PL1': 64000}   |A| = 0.53, |B| = 0.5999 m/rad
   80 cm: {'IL1': 30269, 'IL2': 30553, 'IL3': 25387, 'PL1': 64000}   |A| = 2.24, |B| = 0.8000 m/rad
  100 cm: {'IL1': 30268, 'IL2': 33498, 'IL3': 24034, 'PL1': 64000}   |A| = 1.56, |B| = 0.9999 m/rad


## The ground truth

Written in plain physics: each lens's entrance and length (m, from the specimen) and its
strength (rad of beam rotation, per DAC unit for IL1–PL1), the camera's position, the
deflectors' shift and tilt per DAC unit, the beam at the specimen, the objective's DAC values and
the lens table, all at 200 kV. `SimulatedMicroscope.from_file` builds the microscope from it.

In [5]:
gaps, lengths = np.exp(P[:6]), np.exp(P[6:11])
entrances = np.cumsum(gaps[:5]) + np.r_[0.0, np.cumsum(lengths[:4])]
strength_per_dac = np.asarray(SIGNS) * np.exp(P[12:16]) * STRENGTHS_40K[1:] / np.array([mag_rows[40000][n] for n in LENSES])
fwhm = 2 * np.sqrt(2 * np.log(2))
truth = {
    "description": "The virtual NEOARM, worked out by design.ipynb: hidden lenses fitted to the recorded NEOARM lens "
                   "table, its deflectors, beam and lens table (MAG: the recorded rows by nominal magnification; DIFF: "
                   "designed, by camera length in m), at 200 kV.",
    "reference_voltage_v": 200e3,
    "lenses": {"objective": {"z_m": entrances[0], "length_m": lengths[0], "q": P[11]},
               **{n: {"z_m": z, "length_m": length, "q_per_dac": k}
                  for n, z, length, k in zip(LENSES, entrances[1:], lengths[1:], strength_per_dac)}},
    "camera_z_m": gaps.sum() + lengths.sum(),
    "deflectors": {"shift_m_per_dac": 1e-11, "tilt_rad_per_dac": 2e-7},    # 2000 units: 20 nm; 15000: 3 mrad
    "beam": {"rms_size_m": 40e-9 / fwhm, "rms_angle_rad": 1.1e-3 / fwhm,    # 40 nm wide, 1.1 mrad spread (FWHM)
             "current_a": 1.602176634e-9},                                     # 1e8 electrons in 10 ms
    "objective_dac": objective,
    "lens_table": {"MAG": {f"{m}": row for m, row in mag_rows.items()},
                   "DIFF": {f"{length:g}": row for length, row in diff_rows.items()}},
}
Path("virtual_neoarm.json").write_text(json.dumps(truth, indent=1, default=float) + "\n")
print(json.dumps({k: truth[k] for k in ("lenses", "camera_z_m")}, indent=1, default=float))

{
 "lenses": {
  "objective": {
   "z_m": 0.0005000001122357951,
   "length_m": 0.0002178553101115069,
   "q": -0.6227526140856892
  },
  "IL1": {
   "z_m": 0.05259816951180706,
   "length_m": 0.003645826528171328,
   "q_per_dac": 9.507869169317752e-06
  },
  "IL2": {
   "z_m": 0.3344499129395562,
   "length_m": 0.0031739463431106614,
   "q_per_dac": 2.0906151799124123e-05
  },
  "IL3": {
   "z_m": 0.35762389710632475,
   "length_m": 0.01467009719350765,
   "q_per_dac": 2.5605171123790613e-05
  },
  "PL1": {
   "z_m": 0.6178096169987732,
   "length_m": 0.0011870486290814501,
   "q_per_dac": -7.231643095114933e-06
  }
 },
 "camera_z_m": 0.718996863230155
}
